In [ ]:
# 01_truncated_length_moments.ipynb
# Closed-form cube-truncated fiber-length moments (lambda_1, lambda_2, Var(l_C))
# validated against Monte Carlo, plus the isotropic mean sine E[sin gamma] = pi/4.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "0.3"})
GRAY = "0.35"
FIG_DIR, TAB_DIR = "../results/figures", "../results/tables"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")


rng = np.random.default_rng(20261006)

# --- Model parameters: two-length mixture on the unit cube C = [0, 1]^3 ---
ell = np.array([0.2, 0.4])
p = np.array([0.5, 0.5])
lmax = ell.max()
raw = lambda k: float((p * ell ** k).sum())          # raw length moment E[L^k]
Cvol, Svol = 1.0, (1.0 + lmax) ** 3                   # |C|, |S|

# --- Closed-form truncated moments ---
lam1 = raw(1) * Cvol / Svol
lam2 = (raw(2) - 0.5 * raw(3) + raw(4) / (3 * np.pi) - raw(5) / (40 * np.pi)) / Svol
var_lc = lam2 - lam1 ** 2


def truncated_length(n, rng):
    """Length of a random fiber retained inside the unit cube."""
    L = rng.choice(ell, size=n, p=p)
    X = rng.uniform(-lmax / 2, 1 + lmax / 2, size=(n, 3))
    v = rng.normal(size=(n, 3))
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    lo, hi = -L / 2.0, L / 2.0
    for i in range(3):
        xi, vi = X[:, i], v[:, i]
        zero = np.abs(vi) < 1e-15
        with np.errstate(divide="ignore", invalid="ignore"):
            a, b = (0 - xi) / vi, (1 - xi) / vi
        s_lo = np.where(zero, np.where((xi >= 0) & (xi <= 1), -np.inf, np.inf), np.minimum(a, b))
        s_hi = np.where(zero, np.where((xi >= 0) & (xi <= 1), np.inf, -np.inf), np.maximum(a, b))
        lo, hi = np.maximum(lo, s_lo), np.minimum(hi, s_hi)
    return np.clip(hi - lo, 0.0, None)


lc = truncated_length(2_000_000, rng)

# --- Isotropic mean sine between two independent directions ---
a = rng.normal(size=(1_000_000, 3)); a /= np.linalg.norm(a, axis=1, keepdims=True)
b = rng.normal(size=(1_000_000, 3)); b /= np.linalg.norm(b, axis=1, keepdims=True)
mean_sin = np.sqrt(1 - np.clip((a * b).sum(1), -1, 1) ** 2).mean()

# --- Results: closed form vs Monte Carlo ---
table = pd.DataFrame({
    "quantity": ["lambda_1", "lambda_2", "Var(l_C)", "E[sin gamma]"],
    "closed_form": [lam1, lam2, var_lc, np.pi / 4],
    "monte_carlo": [lc.mean(), (lc ** 2).mean(), lc.var(), mean_sin],
})
print(table.to_string(index=False))
table.to_latex(os.path.join(TAB_DIR, "table_moments.tex"), index=False, float_format="%.6f")

# --- Diagnostic figure: distribution of the cube-truncated length ---
fig, ax = plt.subplots(figsize=(4.2, 3.0))
sns.histplot(lc, bins=60, color=GRAY, stat="density", edgecolor="none", ax=ax)
ax.axvline(lam1, color="black", ls="--", lw=1.2)
ax.set_xlabel(r"$\ell_C$")
ax.set_ylabel("density")
savefig(fig, "fig_ellC_hist")
plt.show()
